# UTS Praktikum Machine Learning

## Agro-Environmental Suitability Classification

Notebook ini digunakan untuk memenuhi tahapan utama project UTS:
- exploratory data analysis (EDA)
- quality check data
- preprocessing dan penanganan imbalance
- ensemble learning dengan Random Forest
- hyperparameter tuning dan cross-validation
- evaluasi model
- export model ke folder `../model`


## Tujuan Eksperimen

Model akan memprediksi `failure_flag` menggunakan empat fitur numerik yang juga dipakai pada aplikasi web:
- `bulk_density`
- `organic_matter_pct`
- `cation_exchange_capacity`
- `salinity_ec`

Pemilihan fitur ini menjaga konsistensi antara proses training di notebook dan proses inferensi di backend.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

DATASET_PATH = Path("../dataset/agro_environmental_dataset.csv")
MODEL_PATH = Path("../model/pipeline.pkl")
FEATURES = [
    "bulk_density",
    "organic_matter_pct",
    "cation_exchange_capacity",
    "salinity_ec",
]
TARGET = "failure_flag"


## Load Dataset dan Cek Awal

Tahap awal mencakup pembacaan dataset, inspeksi dimensi, tipe data, dan beberapa contoh baris untuk memahami struktur data.

In [ ]:
df = pd.read_csv(DATASET_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

df.head()


## Quality Check dan EDA Dasar

Tahap ini mencakup:
- pengecekan missing values
- pengecekan duplikasi
- distribusi target
- ringkasan statistik fitur utama

Walaupun fitur deployment seluruhnya numerik, inspeksi kualitas data tetap dilakukan agar proses modeling lebih valid.

In [ ]:
missing_values = df[FEATURES + [TARGET]].isna().sum()
duplicate_count = df.duplicated().sum()
class_distribution = df[TARGET].value_counts().sort_index()

print("Missing values:\n", missing_values)
print("\nDuplicate rows:", duplicate_count)
print("\nClass distribution:\n", class_distribution)
print("\nClass ratio:\n", (class_distribution / len(df)).round(4))

df[FEATURES].describe().T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x=TARGET, data=df, ax=axes[0])
axes[0].set_title("Distribusi Target")

corr = df[FEATURES + [TARGET]].corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=axes[1])
axes[1].set_title("Korelasi Antar Fitur dan Target")

plt.tight_layout()
plt.show()


## Preprocessing, Split Data, dan Justifikasi Model

Model utama yang dipakai adalah `RandomForestClassifier`. Metode ini dipilih karena:
- termasuk ensemble learning sesuai requirement UTS
- cukup stabil pada data tabular
- mampu menangani hubungan non-linear
- relatif robust terhadap variasi data

Karena target tidak seimbang, pipeline akan menggunakan `SMOTE` saat training. `StandardScaler` tetap dimasukkan ke pipeline agar alur preprocessing terdokumentasi dan konsisten.

In [ ]:
model_df = df[FEATURES + [TARGET]].dropna().copy()

X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain class distribution:\n", y_train.value_counts(normalize=True).round(4))


## Hyperparameter Tuning dan Cross-Validation

Pencarian hyperparameter dilakukan menggunakan `RandomizedSearchCV` dengan `StratifiedKFold` agar evaluasi lebih stabil pada data klasifikasi yang imbalanced.

In [ ]:
pipeline = ImbPipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42,
                class_weight="balanced",
                n_jobs=-1,
            ),
        ),
    ]
)

param_distributions = {
    "classifier__n_estimators": [100, 150, 200, 300],
    "classifier__max_depth": [None, 8, 12, 16, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2", None],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print("Best params:")
print(search.best_params_)
print("\nBest CV F1:", round(search.best_score_, 4))


## Evaluasi Model pada Hold-Out Test Set

Setelah tuning selesai, model terbaik diuji pada data test yang tidak dipakai saat training untuk melihat performa aktual.

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
}

print("Evaluation metrics:")
for name, value in metrics.items():
    print(f"- {name}: {value:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Confusion Matrix - Best Model")
plt.show()


## Cross-Validation Summary pada Model Terbaik

Langkah ini menampilkan ringkasan beberapa metrik validasi silang untuk memastikan model tidak hanya bagus pada satu split data.

In [ ]:
cv_scores = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc",
    },
    n_jobs=-1,
)

cv_summary = pd.DataFrame(cv_scores).agg(["mean", "std"]).T
cv_summary


## Export Model

Model terbaik disimpan ke folder `../model` agar dapat langsung digunakan oleh backend FastAPI.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
print(f"Model saved to: {MODEL_PATH.resolve()}")


## Kesimpulan Singkat

Notebook ini sudah mencakup komponen inti yang diminta pada project UTS:
- EDA dan quality check
- penanganan imbalance dengan SMOTE
- ensemble learning dengan Random Forest
- hyperparameter tuning
- cross-validation
- evaluasi dan export model

Setelah notebook dijalankan, aplikasi backend dan frontend dapat menggunakan model yang tersimpan untuk proses prediksi.